In [1]:
import matplotlib.pyplot as plt
import sys
sys.path.append('../')
import UTILS.utils as utils
import pandas as pd

import numpy as np

## DETERMINING SENTENCE-BASED PARAGRAPH CLASSES

In [2]:
def min_max_normalisation(column):
    min = np.min(column)
    max = np.max(column)
    return (column - min) / (max - min)

In [3]:
all_sentences = pd.read_csv("../../Data/ALL_SENTENCES_CLASSED.csv")

In [5]:
alphabet = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"

def create_classification_prompt(document, topics, topic):
    prompt = "Your task will be to classify the following sentence into one of the following topics, considering that it comes from a paragraph with the general topic '" + topic + "' :"
    prompt += "\nDOCUMENT : {" + document + "}\nTOPICS:\n"
    for i, topic in enumerate(topics):
        prompt += alphabet[i] + " : " + topic + "\n"
    prompt += """
    Your response should be the index of the topic in the list of topics. The format should be only the chosen index and no newlines."""
    return prompt

In [6]:
import google.generativeai as genai

genai.configure(api_key=utils.API_KEY())
model = genai.GenerativeModel('gemini-1.5-flash')


In [7]:
topics = utils.get_list_topics()
letter_to_topic = dict()
for i, t in enumerate(topics):
    letter_to_topic[alphabet[i]] = t

In [8]:
import math
from tqdm import tqdm

def get_sentence_classification():
    sentence_classes = []
    logprobs = []
    paraph_topics = all_sentences["paraph_class_name"].to_list()
    for i, sentence in tqdm(enumerate(all_sentences["text"].to_list())):
        prompt = create_classification_prompt(sentence, topics, paraph_topics[i])
        response = model.generate_content(contents=[prompt])
        sentence_classes.append(letter_to_topic[response.text[0]])
        logprobs.append(math.exp(response.candidates[0].avg_logprobs))
    all_sentences["sentence_paraph_class"] = sentence_classes
    all_sentences["logprob"] = logprobs
    all_sentences.to_csv("ALL_SENTENCES.csv")

#get_sentence_classification()


In [13]:
all_sentences_logprob = pd.read_csv("ALL_SENTENCES.csv", index_col=0)
all_sentences_classed = pd.read_csv("../../Data/ALL_SENTENCES_CLASSED.csv")

In [15]:
sentence_paraph_class = all_sentences_logprob["sentence_paraph_class"].to_list()
logprob = all_sentences_logprob["logprob"].to_list()
all_sentences_classed["sentence_paraph_class"] = sentence_paraph_class
all_sentences_classed["logprob"] = logprob
all_sentences_classed.to_csv("../../Data/ALL_SENTENCES_CLASSED.csv", index=False)


## CREATING 3D COORDINATES

In [4]:
all_sentences_classed = pd.read_csv("../../Data/ALL_SENTENCES_CLASSED.csv")

In [5]:
def get_topic_id_dict(df):
    topics = utils.get_list_topics()
    topic_to_id = dict()
    for topic in topics:
        this_topic = df[df["paraph_class_name"] == topic]
        topic_to_id[topic] = this_topic.sample(1)["paraph_class"].values[0]
    return topic_to_id
        
topic_to_id = get_topic_id_dict(all_sentences_classed)

In [6]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

In [7]:
from umap import UMAP

def get_coordinates(df):
    topics = set(df["paraph_class_name"].to_list())
    topic_to_id = get_topic_id_dict(all_sentences)
    for topic in topics:
        this_topic = df[df["sentence_paraph_class"] == topic]
        this_topic_embeddings = this_topic["text"].apply(lambda x : embedding_model.encode(x))

        ## get x and y coordinates

        umap = UMAP(n_components=2, init='random', random_state=0)
        proj = umap.fit_transform(this_topic_embeddings.to_list())

        x_normal = min_max_normalisation([i[0] for i in proj])
        y_normal = min_max_normalisation([i[1] for i in proj])

        ## get z coordinate

        z_normal = min_max_normalisation(this_topic["logprob"])
        topic_id = topic_to_id[topic]
        z = [i + topic_id for i in z_normal]
        
        df.loc[df["sentence_paraph_class"] == topic, "x"] = x_normal
        df.loc[df["sentence_paraph_class"] == topic, "y"] = y_normal
        df.loc[df["sentence_paraph_class"] == topic, "z"] = z

        

In [8]:
import warnings
warnings.filterwarnings('ignore')

get_coordinates(all_sentences_classed)

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


In [9]:
import plotly.express as px

def display_sentences(df):
    sentences_text = df["text"].to_list()
    N = 13
    result = []
    for text in sentences_text:
        split = text.split(" ")
        b = []
        for i in range(len(split)):
            b.append(split[i])
            if (i + 1) % (N + 1) == 0:
                b.append("<br>")
        split = " ".join(b)
        result.append(split)
    df["hover_text"] = result
    fig_3d = px.scatter_3d(df,x="x", y="y", z="z", color=df["sentence_paraph_class"], hover_data="hover_text")
    fig_3d.update_traces(marker={'size': 7})
    fig_3d.update_layout(legend_title_text='Topic')
    fig_3d.show()


In [10]:
display_sentences(all_sentences_classed)